In [1]:
import math
import random
import pandas as pd

In [2]:
def sigmoid(x):
    x = max(min(x, 500), -500)
    return 1 / (1 + math.exp(-x))

def sigmoid_derivative(y):
    return y * (1 - y)

In [3]:
def initialize_network():
    w1 = [[random.uniform(-0.5, 0.5) for _ in range(3)] for _ in range(3)]
    w2 = [[random.uniform(-0.5, 0.5) for _  in range(1)] for _ in range(3)]
    b_hidden = [random.uniform(-0.5, 0.5) for _ in range(3)]
    b_output = [random.uniform(-0.5, 0.5) for _ in range(1)]
    network = {
        'w1': w1,
        'w2': w2,
        'b_hidden': b_hidden,
        'b_output': b_output
    }
    return network

In [4]:
def forward_pass(network, inputs):
    w1, w2 = network['w1'], network['w2']
    b_hidden, b_output = network['b_hidden'], network['b_output']

    hidden_activations = []
    for i in range(len(w1)):
        z_hidden = sum(inputs[j] * w1[i][j] for j in range(len(inputs))) + b_hidden[i]
        hidden_activations.append(sigmoid(z_hidden))

    z_output = sum(hidden_activations[j] * w2[j][0] for j in range(len(hidden_activations))) + b_output[0]
    output_activation = sigmoid(z_output)
    return hidden_activations, output_activation

In [5]:

def backward_pass(network, hidden_activations, output_activation, target):
    w2 = network['w2']
    output_error = target[0] - output_activation
    output_delta = output_error * sigmoid_derivative(output_activation)

    hidden_deltas = []
    for j in range(len(hidden_activations)):
        error = output_delta * w2[j][0]
        delta = error * sigmoid_derivative(hidden_activations[j])
        hidden_deltas.append(delta)

    return hidden_deltas, output_delta

In [6]:
def update_weights(network, inputs, hidden_activations, hidden_deltas, output_delta, learning_rate):

    for j in range(len(network['w2'])):
        network['w2'][j][0] += learning_rate * output_delta * hidden_activations[j]
    network['b_output'][0] += learning_rate * output_delta
    for i in range(len(network['w1'])):
        for j in range(len(inputs)):
            network['w1'][i][j] += learning_rate * hidden_deltas[i] * inputs[j]
        network['b_hidden'][i] += learning_rate * hidden_deltas[i]

In [7]:
def train_network(network, training_data, epochs, learning_rate):
    for epoch in range(epochs):
        sum_error = 0
        for inputs, target in training_data:
            hidden_activations, output_activation = forward_pass(network, inputs)
            sum_error += (target[0] - output_activation)**2
            hidden_deltas, output_delta = backward_pass(network, hidden_activations, output_activation, target)
            update_weights(network, inputs, hidden_activations, hidden_deltas, output_delta, learning_rate)
            
        if epoch % 1000 == 0 or epoch == epochs - 1:
            print(f'> Epoch={epoch}, Learning Rate={learning_rate:.2f}, Error={sum_error:.4f}')

In [8]:
BMI = pd.read_csv('../bmi.csv')
BMI

,Gender,Height,Weight,Index
0,Male,174,96,4
1,Male,189,87,2
2,Female,185,110,4
3,Female,195,104,3
4,Male,149,61,3
...,...,...,...,...
495,Female,150,153,5
496,Female,184,121,4
497,Female,141,136,5
498,Male,150,95,5


In [9]:
BMI['Gender'] = BMI['Gender'].apply(lambda x: 1 if str(x).lower() == 'female' else 0)
BMI['BMI'] = BMI['Weight'] / ((BMI['Height'] / 100) ** 2)
norm_params = {
    'Height': {'mean': BMI['Height'].mean(), 'std': BMI['Height'].std()},
    'Weight': {'mean': BMI['Weight'].mean(), 'std': BMI['Weight'].std()}
}
scale_params = {
    'BMI': {'min': BMI['BMI'].min(), 'max': BMI['BMI'].max()}
}

In [10]:
if __name__ == "__main__":

    training_dataset = []
    for _, row in BMI.iterrows():
        inputs = [
            row['Gender'],
            (row['Height'] - norm_params['Height']['mean']) / norm_params['Height']['std'],
            (row['Weight'] - norm_params['Weight']['mean']) / norm_params['Weight']['std']
        ]
        bmi_val = row['BMI']
        scaled_bmi = 0.1 + 0.8 * (bmi_val - scale_params['BMI']['min']) / (scale_params['BMI']['max'] - scale_params['BMI']['min'])
        target = [scaled_bmi]
        training_dataset.append((inputs, target))

    network = initialize_network()
    learning_rate = 0.01
    epochs = 10000
    
    print("--- Starting Network Training ---")
    train_network(network, training_dataset, epochs+1, learning_rate)
    print("--- Training Complete ---")

--- Starting Network Training ---
> Epoch=0, Learning Rate=0.01, Error=20.1475
> Epoch=1000, Learning Rate=0.01, Error=0.1712
> Epoch=2000, Learning Rate=0.01, Error=0.1098
> Epoch=3000, Learning Rate=0.01, Error=0.0922
> Epoch=4000, Learning Rate=0.01, Error=0.0797
> Epoch=5000, Learning Rate=0.01, Error=0.0696
> Epoch=6000, Learning Rate=0.01, Error=0.0608
> Epoch=7000, Learning Rate=0.01, Error=0.0526
> Epoch=8000, Learning Rate=0.01, Error=0.0448
> Epoch=9000, Learning Rate=0.01, Error=0.0375
> Epoch=10000, Learning Rate=0.01, Error=0.0309
--- Training Complete ---


In [11]:
print("\n--- Making a Prediction ---")
test_gender = 0
test_height_cm = 174
test_weight_kg = 96

norm_height = (test_height_cm - norm_params['Height']['mean']) / norm_params['Height']['std']
norm_weight = (test_weight_kg - norm_params['Weight']['mean']) / norm_params['Weight']['std']
test_input = [test_gender, norm_height, norm_weight]
_, scaled_prediction = forward_pass(network, test_input)

unscaled_pred = (scaled_prediction - 0.1) / 0.8
predicted_bmi = unscaled_pred * (scale_params['BMI']['max'] - scale_params['BMI']['min']) + scale_params['BMI']['min']

actual_bmi = test_weight_kg / ((test_height_cm / 100)**2)

print(f"Input: Gender=Male, Height={test_height_cm}cm, Weight={test_weight_kg}kg")
print(f"Predicted BMI: {predicted_bmi:.2f}")
print(f"Actual BMI for comparison: {actual_bmi:.2f}")


--- Making a Prediction ---
Input: Gender=Male, Height=174cm, Weight=96kg
Predicted BMI: 31.07
Actual BMI for comparison: 31.71
